In [1]:
import numpy as np
import matplotlib.pyplot as plt# Later: load, align (if needed), and plot ΔP = P_B − P_A
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt
from scipy.fft import fft, fftfreq
from scipy.integrate import simpson  # For numerical integration (Simpson's rule)
from scipy.integrate import quad


def extract_envelopes(signal, peak_distance=None, smooth_cutoff_freq=None, sampling_freq=1.0):
    """
    Extracts the upper (max) and lower (min) envelopes of an oscillating signal.
    
    Parameters:
    - signal: np.ndarray, the input numerical array (pressure values).
    - peak_distance: int or None, minimum distance between peaks (in indices). If None, auto-detect based on dominant frequency.
                     For auto-detection, it's set to approximately half the dominant period to handle variations.
    - smooth_cutoff_freq: float or None, cutoff frequency for low-pass smoothing of envelopes.
                          If None, no smoothing is applied. Should be much lower than oscillation frequency.
    - sampling_freq: float, sampling frequency of the signal (for frequency detection and smoothing filter).
    
    Returns:
    - upper_envelope: np.ndarray, the upper envelope (max) as a function of index.
    - lower_envelope: np.ndarray, the lower envelope (min) as a function of index.
    """
    if not isinstance(signal, np.ndarray):
        signal = np.array(signal)
    
    indices = np.arange(len(signal))
    
    if peak_distance is None:
        # Auto-detect dominant frequency using FFT
        fft_vals = np.fft.fft(signal)
        freqs = np.fft.fftfreq(len(signal), 1 / sampling_freq)
        # Ignore DC component (freq=0), find peak frequency
        positive_freq_idx = freqs > 0
        if np.any(positive_freq_idx):
            idx = np.argmax(np.abs(fft_vals[positive_freq_idx]))
            dominant_freq = freqs[positive_freq_idx][idx]
            if dominant_freq > 0:
                period_samples = sampling_freq / dominant_freq
                peak_distance = int(period_samples / 2)  # Set to half period for robustness (allows detection with some variation)
            else:
                peak_distance = 10  # Fallback if no dominant frequency detected
        else:
            peak_distance = 10  # Fallback for very short signals
    
    # Ensure peak_distance is at least 1
    peak_distance = max(1, peak_distance)
    
    # Find local maxima for upper envelope
    max_peaks, _ = find_peaks(signal, distance=peak_distance)
    
    # Find local minima for lower envelope
    min_peaks, _ = find_peaks(-signal, distance=peak_distance)
    
    # Interpolate upper envelope
    if len(max_peaks) > 1:
        upper_interp = interp1d(indices[max_peaks], signal[max_peaks], kind='linear', fill_value='extrapolate')
        upper_envelope = upper_interp(indices)
    else:
        upper_envelope = np.full_like(signal, np.max(signal) if len(signal) > 0 else np.nan)
    
    # Interpolate lower envelope
    if len(min_peaks) > 1:
        lower_interp = interp1d(indices[min_peaks], signal[min_peaks], kind='linear', fill_value='extrapolate')
        lower_envelope = lower_interp(indices)
    else:
        lower_envelope = np.full_like(signal, np.min(signal) if len(signal) > 0 else np.nan)
    
    # Optional: Smooth the envelopes with a low-pass filter
    if smooth_cutoff_freq is not None:
        nyq = 0.5 * sampling_freq
        normal_cutoff = smooth_cutoff_freq / nyq
        b, a = butter(3, normal_cutoff, btype='low', analog=False)
        upper_envelope = filtfilt(b, a, upper_envelope)
        lower_envelope = filtfilt(b, a, lower_envelope)
    
    return upper_envelope, lower_envelope


import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

def find_wavepacket_envelope(t, x, method='peaks'):
    """
    Find the upper and lower envelopes of an oscillating wavepacket.
    
    Parameters:
    -----------
    t : array-like
        Time array
    x : array-like
        Signal array (oscillating wavepacket)
    method : str, optional
        Method to use: 'peaks' (default) or 'hilbert'
        
    Returns:
    --------
    t_envelope : array
        Time points for envelope (same as input t for hilbert method)
    upper_envelope : array
        Upper envelope values
    lower_envelope : array
        Lower envelope values
    """
    
    t = np.asarray(t)
    x = np.asarray(x)
    
    if method == 'peaks':
        # Find local maxima and minima
        peaks_max, _ = find_peaks(x)
        peaks_min, _ = find_peaks(-x)
        
        # Add endpoints to ensure envelope spans entire range
        if len(peaks_max) > 0:
            peaks_max = np.concatenate([[0], peaks_max, [len(x)-1]])
        else:
            peaks_max = np.array([0, len(x)-1])
            
        if len(peaks_min) > 0:
            peaks_min = np.concatenate([[0], peaks_min, [len(x)-1]])
        else:
            peaks_min = np.array([0, len(x)-1])
        
        # Remove duplicate indices
        peaks_max = np.unique(peaks_max)
        peaks_min = np.unique(peaks_min)
        
        # Interpolate to create smooth envelopes
        upper_interp = interp1d(t[peaks_max], x[peaks_max], 
                               kind='cubic', fill_value='extrapolate')
        lower_interp = interp1d(t[peaks_min], x[peaks_min], 
                               kind='cubic', fill_value='extrapolate')
        
        upper_envelope = upper_interp(t)
        lower_envelope = lower_interp(t)
        t_envelope = t
        
    elif method == 'hilbert':
        # Use Hilbert transform to get analytical signal
        from scipy.signal import hilbert
        
        analytic_signal = hilbert(x)
        amplitude_envelope = np.abs(analytic_signal)
        
        upper_envelope = amplitude_envelope
        lower_envelope = -amplitude_envelope
        t_envelope = t
    
    else:
        raise ValueError("method must be 'peaks' or 'hilbert'")
    
    return t_envelope, upper_envelope, lower_envelope